# 03 · Event Extraction

**Purpose:** Label each (article, ticker) pair with structured event types using
rule-based pattern matching, plus negation and speculation detection.

**Event types:** `ma` · `earnings` · `management` · `legal`

**Input:** `articles_mapped.parquet`  
**Output:** `events.parquet`  
Schema: `article_url, ticker, date, event_ma, event_earnings, event_management, event_legal, is_negated, is_speculative`

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [2]:
articles_mapped = load("articles_mapped")
print(f"{len(articles_mapped):,} (article, ticker) pairs")

3,557 (article, ticker) pairs


## 1 · Pattern dictionaries

In [3]:
# Each list entry is a compiled regex pattern
EVENT_PATTERNS: dict[str, list[re.Pattern]] = {
    "ma": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bacquisition\b", r"\bfusion\b", r"\brachat\b",
            r"\bOPA\b", r"\bcession\b", r"\boffre publique\b",
            r"\bprise de participation\b", r"\bmerger\b",
            r"\btakeover\b", r"\bconsortium\b",
        ]
    ],
    "earnings": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\brésultats?\b", r"\bbénéfice[s]?\b", r"\bEBIT\b",
            r"\bEBITDA\b", r"\bchiffre d.affaires\b", r"\brevenus?\b",
            r"\bdividende[s]?\b", r"\bbénéfice net\b", r"\bmarge\b",
            r"\bperte[s]?\b", r"\bprofit[s]?\b", r"\bearnings\b",
        ]
    ],
    "management": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bPDG\b", r"\bdirecteur.général\b", r"\bnomination\b",
            r"\bdémission\b", r"\bconseil d.administration\b",
            r"\bprésidence\b", r"\bnommé\b", r"\bCEO\b",
            r"\bretraite\b", r"\bcomité exécutif\b",
        ]
    ],
    "legal": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bprocès\b", r"\blitige\b", r"\bsanction\b",
            r"\bamende\b", r"\btribunal\b", r"\brégulateur\b",
            r"\bpoursuite[s]?\b", r"\bplainte\b", r"\benquête\b",
            r"\bfraud[e]?\b", r"\bcorruption\b",
        ]
    ],
}

NEGATION_PATTERNS = [
    re.compile(p, re.IGNORECASE) for p in [
        r"\bne\b.{0,25}\bpas\b", r"\baucun\b", r"\bpas de\b",
        r"\bn.a pas\b", r"\bn.ont pas\b", r"\bjamais\b", r"\bsans\b",
    ]
]

SPECULATION_PATTERNS = [
    re.compile(p, re.IGNORECASE) for p in [
        r"\bpourrait\b", r"\benvisage\b", r"\bprévoit\b",
        r"\bselon des sources\b", r"\brumeur\b", r"\bdevrait\b",
        r"\bserait\b", r"\bnégociation[s]?\b", r"\bdiscussion[s]?\b",
        r"\bà l.étude\b", r"\bselon nos informations\b",
    ]
]

print("Patterns loaded.")

Patterns loaded.


## 2 · Detection logic

In [4]:
def _any_match(text: str, patterns: list[re.Pattern]) -> bool:
    return any(p.search(text) for p in patterns)


def extract_events(text: str) -> dict:
    row: dict = {}
    for event_type, patterns in EVENT_PATTERNS.items():
        row[f"event_{event_type}"] = _any_match(text, patterns)
    row["is_negated"]    = _any_match(text, NEGATION_PATTERNS)
    row["is_speculative"] = _any_match(text, SPECULATION_PATTERNS)
    return row

## 3 · Apply to dataset

In [5]:
event_cols = articles_mapped["text"].apply(extract_events).apply(pd.Series)

events = pd.concat(
    [articles_mapped[["article_url", "ticker", "date", "published_at"]].reset_index(drop=True),
     event_cols.reset_index(drop=True)],
    axis=1,
)

bool_cols = ["event_ma", "event_earnings", "event_management", "event_legal",
             "is_negated", "is_speculative"]

print("Event coverage (% of (article, ticker) pairs):")
print((events[bool_cols].mean() * 100).round(1).to_string())

Event coverage (% of (article, ticker) pairs):
event_ma             1.8
event_earnings      18.5
event_management    10.0
event_legal          4.2
is_negated          26.0
is_speculative      15.1


In [ ]:

has_any = events[bool_cols[:4]].any(axis=1)
print(f"Rows with ≥1 event:  {has_any.sum():,} / {len(events):,} ({has_any.mean()*100:.1f}%)")
# Co-occurrence matrix
events[bool_cols[:4]].astype(int).corr().round(2)

Rows with ≥1 event:  1,122 / 3,557 (31.5%)


,event_ma,event_earnings,event_management,event_legal
event_ma,1.00,0.00,0.05,-0.03
event_earnings,0.00,1.00,-0.06,0.01
event_management,0.05,-0.06,1.00,-0.05
event_legal,-0.03,0.01,-0.05,1.00


In [7]:
save(events, "events")

  saved 3,557 rows  →  events.parquet


WindowsPath('C:/_PROJECTS/pfa_bvc/Notebooks/signal_pipeline/data/events.parquet')